# Lab: put a guard on an agent, with LangGraph

A real agent (DeepSeek, through OpenRouter) runs the music app. It just got a new tool, `delete_playlist`. First you watch it do something it should never do **without asking**. Then you build the guard with LangGraph: a **router** (which tool needs a human?) and an **approval step** that pauses the run, even across a process restart.

**Setup:** put `OPENROUTER_API_KEY=...` in a `.env` file at the repository root (see `.env.example`); never commit it. Run the cells in order. `lab.py` is given code: no need to read it.

**You write:** seven blanks, written `___`, in two cells (sections 2 and 3). A test right after each cell tells you which blank is wrong.

In [ ]:
import lab                                # given: the music app, the model, the graph, the helpers
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END
from langgraph.types import Command, interrupt

lab.check_setup()
lab.reset()
print("The library today:"); lab.show_library()

## 1. The unguarded agent

The graph below is a loop: **`think`** (one model call) then the **router** picks the next node. This router has no guard: whatever the model asks for, it runs. The user says: *"I am bored of my Focus playlist. Delete it so I can start fresh."*

In [ ]:
app = lab.build_graph(lab.unguarded_router)
lab.show_graph(app)

In [ ]:
PRANK = "I am bored of my Focus playlist. Delete it so I can start fresh."

lab.ask(app, PRANK)

The playlist is **gone**, for good, and nobody was asked. A prompt cannot fix that: the model is not the guard. The guard is code around the model, and that is what a graph is for.

## 2. Fill in the router

The router is a plain function: state in, next node out. The order of the checks is given. Replace the four `___`: the tool that needs a human, then the next node for three cases (`"give_up"`, `"approve"` or `"run_tool"`). Then run the test: it uses **no model**.

In [ ]:
NEEDS_APPROVAL = {"___"}                  # blank 1: a tool name (search_tracks, get_track, add_to_playlist, delete_playlist)


def router(state):
    """After the model has spoken, return the name of the next node."""
    if state["action"]["type"] == "final":
        return END
    if state["steps"] >= state["max_steps"]:
        return "___"                      # blank 2: the bound is reached, stop the run
    if state["action"]["tool"] in NEEDS_APPROVAL:
        return "___"                      # blank 3: a human must decide first
    return "___"                          # blank 4: a harmless tool, just run it

In [ ]:
def state(action, steps=1): return {"action": action, "steps": steps, "max_steps": 4}
search = {"type": "call", "tool": "search_tracks", "args": {"query": "Night"}}
delete = {"type": "call", "tool": "delete_playlist", "args": {"playlist": "Focus"}}

assert router(state({"type": "final"})) == END,          "a final answer ends the run"
assert router(state(search)) == "run_tool",              "a harmless tool just runs (blank 4)"
assert router(state(delete)) == "approve",               "a delete goes to a human (blanks 1 and 3)"
assert router(state(search, steps=4)) == "give_up",      "at the bound, stop (blank 2)"
assert router(state(delete, steps=4)) == "give_up",      "a delete at the bound must NOT run"
assert router(state({"type": "final"}, steps=4)) == END, "a final answer at the bound is still an answer"
print("OK: the router is right, tested with no model, no tokens, no money")

## 3. Fill in the approval step

`interrupt()` stops the run **and saves it** in a checkpoint. Whoever resumes it gives back a decision, which `interrupt()` returns: a dictionary with the keys `approve` (true or false) and `reason`. Replace the three `___`. One rule, already respected by the given lines: on resume this function restarts from its first line, so nothing before `interrupt()` may have an effect.

In [ ]:
def approve(state):
    """Pause until a human decides. The graph comes here instead of running a tool that needs approval."""
    action = state["action"]
    decision = interrupt({"tool": action["tool"], "args": action["args"]})    # the run stops here
    if decision["___"]:                                    # blank 5: the key that says yes or no
        return Command(goto="___")                         # blank 6: approved, run the tool
    rejected = {"tool": action["tool"], "args": action["args"],
                "result": "rejected by the human: " + decision.get("reason", "")}
    return Command(goto="___", update={"observations": [rejected]})    # blank 7: the model reads the rejection

In [ ]:
lab.check_approve(approve)             # no model, no human: both answers are tested

## 4. The same prank, now guarded

Same request, same model, a different graph. `InMemorySaver` is the checkpointer: it remembers where the run stopped.

In [ ]:
lab.reset()
app = lab.build_graph(router, approve, InMemorySaver())
lab.show_graph(app)
lab.ask(app, PRANK, thread="t1")

The run **stopped before the delete**: the playlists are untouched and the payload shows what the human would approve. Now say no.

In [ ]:
lab.ask(app, Command(resume={"approve": False, "reason": "I changed my mind"}), thread="t1")

The model read the rejection as the tool's result and carried on. Now a second run, and say yes.

In [ ]:
lab.ask(app, PRANK, thread="t2")
lab.ask(app, Command(resume={"approve": True}), thread="t2")

## 5. Kill the process, approve from a new one

The pause is saved in a SQLite file, not in memory. The cell below runs the same graph in **one process**, which pauses and exits, then in a **second process**, which resumes it. Look at the process ids and at the number of model calls.

In [ ]:
lab.restart_demo(router, approve)

The second process did not start over: it loaded the checkpoint, got a "yes", ran the delete **once**, and called the model **one more time**, only to word the answer (compare the two counts).

## 6. Bonus: two safety nets for a stuck agent

An impossible task: *"find a track that does not exist, never give up"*. Net 1 is LangGraph's `recursion_limit`: it raises an exception. Net 2 is **your** `max_steps`, checked in your router: the run ends cleanly, with a status.

In [ ]:
from langgraph.errors import GraphRecursionError

lab.reset()
app = lab.build_graph(router)
try:
    app.invoke({"question": lab.IMPOSSIBLE, "max_steps": 99}, {"recursion_limit": 10})
except GraphRecursionError as e:
    print("net 1, LangGraph:", type(e).__name__, "after", lab.model_calls(), "model calls")

lab.reset()
out = app.invoke({"question": lab.IMPOSSIBLE, "max_steps": 4}, {"recursion_limit": 25})
print("net 2, your router: status =", out["status"], "after", lab.model_calls(), "model calls")

**Questions:** why must the delete at the bound go to `give_up` and not to `approve`? What would the second process of section 5 have done if `approve` had run code with an effect *before* `interrupt()`?